# 7-Eleven AI Functions Demo – Data Setup

This notebook creates and populates **7-Eleven-relevant demo data** for showcasing Databricks AI Functions to data analysts.

**Tables created:**
- **store_info** – Store locations and attributes (for ai_query, joins)
- **daily_sales** – Store-level sales by category (for ai_query)
- **customer_reviews** – Unstructured review text (for ai_summarize, ai_analyze_sentiment, ai_extract, ai_classify)
- **product_catalog** – Product names and descriptions (for ai_gen, ai_similarity)
- **feedback_notes** – Support/feedback text (for ai_classify)

**Instructions:**
1. Set `CATALOG` and `SCHEMA` below to your Unity Catalog location.
2. Run all cells in order.
3. Proceed to **02_ai_functions_demo** to run the AI function examples.

In [ ]:
-- Configuration: set your catalog and schema
SET VAR catalog_name = 'renjiharold_demo';
SET VAR schema_name = 'ai_functions_demo';

CREATE SCHEMA IF NOT EXISTS ${catalog_name}.${schema_name};

SELECT 'Schema ready: ' || current_schema() AS status;

## 1. Store information (locations, regions)

In [ ]:
CREATE OR REPLACE TABLE ${catalog_name}.${schema_name}.store_info (
  store_id STRING,
  city STRING,
  state STRING,
  region STRING,
  store_type STRING,
  open_date DATE
) USING DELTA;

INSERT INTO ${catalog_name}.${schema_name}.store_info VALUES
('ST001', 'Dallas', 'TX', 'South', 'Standard', '2020-01-15'),
('ST002', 'Houston', 'TX', 'South', 'Standard', '2019-06-01'),
('ST003', 'Austin', 'TX', 'South', 'Express', '2021-03-10'),
('ST004', 'Phoenix', 'AZ', 'Southwest', 'Standard', '2018-11-20'),
('ST005', 'San Diego', 'CA', 'West', 'Standard', '2020-09-05'),
('ST006', 'Los Angeles', 'CA', 'West', 'Express', '2017-04-12'),
('ST007', 'Chicago', 'IL', 'Midwest', 'Standard', '2019-08-22'),
('ST008', 'Denver', 'CO', 'Mountain', 'Standard', '2022-01-30'),
('ST009', 'Miami', 'FL', 'Southeast', 'Standard', '2018-05-14'),
('ST010', 'Atlanta', 'GA', 'Southeast', 'Express', '2021-07-01');

SELECT * FROM ${catalog_name}.${schema_name}.store_info;

## 2. Daily sales by store and category (for ai_query)

In [ ]:
CREATE OR REPLACE TABLE ${catalog_name}.${schema_name}.daily_sales (
  sale_date DATE,
  store_id STRING,
  category STRING,
  revenue DOUBLE,
  units_sold INT
) USING DELTA;

INSERT INTO ${catalog_name}.${schema_name}.daily_sales
SELECT sale_date, store_id, category, revenue, units_sold FROM (VALUES
  ('2024-11-01', 'ST001', 'Beverages', 450.25, 120), ('2024-11-01', 'ST001', 'Snacks', 320.50, 85),
  ('2024-11-01', 'ST001', 'Prepared Food', 580.00, 95), ('2024-11-01', 'ST002', 'Beverages', 510.75, 140),
  ('2024-11-01', 'ST002', 'Slurpee', 290.00, 72), ('2024-11-02', 'ST001', 'Beverages', 475.00, 125),
  ('2024-11-02', 'ST003', 'Prepared Food', 620.30, 88), ('2024-11-02', 'ST004', 'Fuel', 1200.00, 0),
  ('2024-11-03', 'ST005', 'Beverages', 380.00, 98), ('2024-11-03', 'ST006', 'Snacks', 410.20, 110),
  ('2024-11-04', 'ST007', 'Beer & Wine', 890.50, 45), ('2024-11-04', 'ST008', 'Prepared Food', 540.00, 78),
  ('2024-11-05', 'ST009', 'Dairy', 220.00, 55), ('2024-11-05', 'ST010', 'Beverages', 495.00, 130),
  ('2024-11-06', 'ST001', 'Slurpee', 310.00, 80), ('2024-11-06', 'ST002', 'Prepared Food', 590.00, 92),
  ('2024-11-07', 'ST004', 'Snacks', 355.00, 88), ('2024-11-07', 'ST006', 'Beverages', 520.00, 135),
  ('2024-11-08', 'ST008', 'Beverages', 440.00, 115), ('2024-11-09', 'ST010', 'Prepared Food', 610.00, 90)
) AS t(sale_date, store_id, category, revenue, units_sold);

SELECT sale_date, store_id, category, revenue, units_sold
FROM ${catalog_name}.${schema_name}.daily_sales
ORDER BY sale_date DESC, store_id
LIMIT 30;

## 3. Customer reviews (unstructured text for summarize, sentiment, extract, classify)

In [ ]:
CREATE OR REPLACE TABLE ${catalog_name}.${schema_name}.customer_reviews (
  review_id STRING,
  store_id STRING,
  review_text STRING,
  rating INT,
  created_at DATE
) USING DELTA;

INSERT INTO ${catalog_name}.${schema_name}.customer_reviews VALUES
('R001', 'ST001', 'Stopped in for coffee and a breakfast taquito. Coffee was fresh and the taquito was hot. Staff was friendly. Only downside was the bathroom was out of order which was inconvenient.', 4, '2024-11-01'),
('R002', 'ST002', 'Terrible experience. Slurpee machine broken, shelves were half empty, and the cashier was rude. I had to wait 10 minutes for someone to unlock the restroom. Will not be back.', 1, '2024-11-05'),
('R003', 'ST003', 'Quick in and out. Grabbed a Big Gulp and some chips. Prices are a bit high but the store is clean and well lit. No complaints.', 4, '2024-11-08'),
('R004', 'ST004', 'Love the new hot food section. The pizza and roller grill items are actually really good. Store in Phoenix on 7th St is always stocked. Employees remember regulars.', 5, '2024-11-10'),
('R005', 'ST005', 'Needed gas and a snack. Fuel price was competitive. Inside was clean. Bought a 7-Select water and a banana. Checkout was fast. Neutral experience overall.', 3, '2024-11-12'),
('R006', 'ST006', 'The Slurpee selection here is amazing. My kids love it. Sometimes the line is long but the staff moves people through quickly. Parking can be tight in the evening.', 4, '2024-11-15'),
('R007', 'ST007', 'Disappointed. Came in for the 2 for $5 deal on energy drinks but they were sold out. Shelves looked disorganized. One positive: the clerk was very apologetic and offered a rain check.', 3, '2024-11-18'),
('R008', 'ST008', 'Best 7-Eleven in Denver. Always has fresh coffee, clean restrooms, and the prepared sandwiches are way better than I expected. Manager runs a tight ship.', 5, '2024-11-20'),
('R009', 'ST009', 'Mixed feelings. Great location and the store has a good selection of beer and wine. But the cold vault was warm and several items were out of date. Reported it to staff.', 2, '2024-11-22'),
('R010', 'ST010', 'Stopped for gas and ended up buying a whole meal. The hot dogs and nachos hit the spot. Store was busy but well organized. Will definitely return.', 5, '2024-11-25');

SELECT review_id, store_id, LEFT(review_text, 80) || '...' AS review_preview, rating, created_at
FROM ${catalog_name}.${schema_name}.customer_reviews;

## 4. Product catalog (for ai_gen and ai_similarity)

In [ ]:
CREATE OR REPLACE TABLE ${catalog_name}.${schema_name}.product_catalog (
  product_id STRING,
  product_name STRING,
  category STRING,
  short_description STRING
) USING DELTA;

INSERT INTO ${catalog_name}.${schema_name}.product_catalog VALUES
('P001', 'Big Gulp Cola', 'Beverages', '32 oz fountain cola'),
('P002', 'Big Gulp Diet Cola', 'Beverages', '32 oz diet fountain cola'),
('P003', 'Slurpee Cherry', 'Beverages', 'Frozen cherry flavored drink'),
('P004', 'Slurpee Blue Raspberry', 'Beverages', 'Frozen blue raspberry drink'),
('P005', '7-Select Sparkling Water', 'Beverages', '12 oz sparkling water variety pack'),
('P006', 'Taquito Chicken', 'Prepared Food', 'Rolled tortilla with chicken filling'),
('P007', 'Taquito Beef', 'Prepared Food', 'Rolled tortilla with beef filling'),
('P008', 'Pizza Slice Pepperoni', 'Prepared Food', 'Single slice pepperoni pizza'),
('P009', 'Hot Dog All-American', 'Prepared Food', 'Classic hot dog with condiments'),
('P010', 'Lay''s Classic Chips', 'Snacks', 'Classic potato chips 1.5 oz'),
('P011', 'Doritos Nacho Cheese', 'Snacks', 'Nacho cheese flavored tortilla chips'),
('P012', '7-Select Trail Mix', 'Snacks', 'Mixed nuts and dried fruit'),
('P013', 'Red Bull Original', 'Beverages', '8.4 oz energy drink'),
('P014', 'Monster Energy Green', 'Beverages', '16 oz energy drink'),
('P015', 'Banana', 'Grocery', 'Single banana');

SELECT * FROM ${catalog_name}.${schema_name}.product_catalog;

## 5. Feedback / support notes (for ai_classify)

In [ ]:
CREATE OR REPLACE TABLE ${catalog_name}.${schema_name}.feedback_notes (
  note_id STRING,
  store_id STRING,
  note_text STRING,
  created_at DATE
) USING DELTA;

INSERT INTO ${catalog_name}.${schema_name}.feedback_notes VALUES
('N001', 'ST001', 'Customer called to say the coffee machine was dispensing cold coffee all morning. We checked and the heating element was faulty. Replaced by 11am.', '2024-11-02'),
('N002', 'ST002', 'Multiple complaints about dirty restrooms and out-of-stock Slurpee flavors. Scheduling deep clean and ordering more syrup.', '2024-11-06'),
('N003', 'ST004', 'Positive feedback: customer said our new hot food display is the best in the region and asked to speak to the manager to compliment the team.', '2024-11-11'),
('N004', 'ST006', 'Employee reported that the cooler door was left open overnight. Some product had to be written off. Reminder sent to closing crew about walk-through checklist.', '2024-11-16'),
('N005', 'ST007', 'Vendor delivery was short 5 cases of energy drinks. Customer asked for 2-for-5 deal and we had to turn them away. Follow-up with distributor.', '2024-11-19'),
('N006', 'ST009', 'Customer found expired yogurt in cold vault. We pulled all dairy and did date check. Two other items were out of date. Staff retrained on rotation.', '2024-11-23'),
('N007', 'ST010', 'Great review on Google – customer said our hot dogs and nachos were fresh and the store was the cleanest they have seen. Shared with team.', '2024-11-26');

SELECT note_id, store_id, LEFT(note_text, 100) || '...' AS note_preview, created_at
FROM ${catalog_name}.${schema_name}.feedback_notes;

## 6. Verify all tables (optional)

In [ ]:
SHOW TABLES IN ${catalog_name}.${schema_name};

SELECT 'store_info' AS tbl, COUNT(*) AS cnt FROM ${catalog_name}.${schema_name}.store_info
UNION ALL
SELECT 'daily_sales', COUNT(*) FROM ${catalog_name}.${schema_name}.daily_sales
UNION ALL
SELECT 'customer_reviews', COUNT(*) FROM ${catalog_name}.${schema_name}.customer_reviews
UNION ALL
SELECT 'product_catalog', COUNT(*) FROM ${catalog_name}.${schema_name}.product_catalog
UNION ALL
SELECT 'feedback_notes', COUNT(*) FROM ${catalog_name}.${schema_name}.feedback_notes;